# 01. 데이터 검사와 고정 split 만들기

담당: 수빈
목표: CSV와 이미지를 검사하고, 근접 중복 후보를 사람이 직접 확인한 다음, 팀 전체가 공유할 고정
validation split을 만들어 내용 해시와 함께 배포하는 것입니다.

이 split이 확정되어야 창혁의 프롬프트 비교와 유림의 학습이 시작될 수 있습니다. 계획서 기준으로
G1 게이트에 해당하므로, 다른 사람이 기다리고 있다는 점을 염두에 두고 진행하십시오.

## 통과 기준(PASS)
1. `scripts/audit_data.py` 가 종료 코드 0으로 끝나고, 경로 누락과 손상 이미지가 0건입니다.
2. 근접 중복 후보를 사람이 눈으로 확인해서 같은 장면인지 여부를 기록했습니다.
3. val 문항 수가 300에서 500 사이이고, 정답 a~d 분포가 한쪽으로 크게 치우치지 않습니다.
4. train_ids와 val_ids가 서로 겹치지 않고, 그룹이 양쪽으로 쪼개지지 않았습니다.

## 넘겨줄 내용
마지막 셀이 출력하는 split 해시 블록을 팀 채널에 공유합니다. 전원이 같은 split을 쓰는지 확인하는
기준값이 됩니다.

### 명령 인자에 대한 안내

이 노트북의 명령은 `vqa/` 와 `scripts/` 에 실제로 구현된 인자에 맞춰 작성했습니다. 그래도 코어
코드가 계속 바뀌는 동안에는 인자 이름이 달라질 수 있습니다. 셀이 `unrecognized arguments` 로
실패하면 먼저 해당 명령에 `--help` 를 붙여서 실제 인자 이름을 확인하고, 셀 안의 문자열만 고치면
됩니다. 노트북의 다른 부분은 고치지 않아도 됩니다. 고친 내용은 팀 채널에 알려서 다른 사람이 같은
곳에서 막히지 않게 하십시오.

In [ ]:
# ===== 파라미터 =====
PROJECT_ROOT = r"."
DATA_ROOT = r"."
RUN_ROOT = r"./artifacts/runs"
LOCAL_SCRATCH = r"./artifacts/scratch"
MODEL_ROOT = r""

OUT_DIR = "artifacts/data"
AUDIT_REPORT = "artifacts/data/audit_report.json"
PAIRS_CSV = "artifacts/data/near_duplicate_pairs.csv"
SPLIT_OUT = "artifacts/data/split_v1.json"
VAL_SIZE = 400          # 계획서의 300~500 목표 범위 안에서 정합니다.
SEED = 20250921
PHASH_THRESHOLD = 8     # 근접 중복 판정에 쓸 perceptual hash 거리 상한
DUP_PAIRS_TO_SHOW = 24  # 한 번에 눈으로 확인할 근접 중복 후보 쌍의 개수

In [ ]:
# --- 공통 부트스트랩: 경로, 환경 변수, 명령 실행 헬퍼 (수정할 일 거의 없음) ---
import os, sys, json, shlex, subprocess, time, csv, hashlib, random
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def _setenv(name, value, create=False):
    if not value:
        return None
    p = Path(value).expanduser()
    if create:
        p.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(p)
    return p


DATA_ROOT_P = _setenv("VQA_DATA_ROOT", DATA_ROOT)
RUN_ROOT_P = _setenv("VQA_RUN_ROOT", RUN_ROOT, create=True)
SCRATCH_P = _setenv("VQA_LOCAL_SCRATCH", LOCAL_SCRATCH, create=True)
MODEL_ROOT_P = _setenv("VQA_MODEL_ROOT", MODEL_ROOT)

PY = sys.executable


def run(cmd, check=True, echo=True, quiet=False):
    """명령을 실행하면서 출력을 그대로 보여 주고 종료 코드와 소요 시간을 돌려줍니다."""
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    cmd = [str(x) for x in cmd]
    if echo:
        print("$ " + " ".join(shlex.quote(x) for x in cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        cmd, cwd=str(PROJECT_ROOT), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        errors="replace", bufsize=1,
    )
    buf = []
    for line in proc.stdout:
        buf.append(line)
        if not quiet:
            print(line, end="", flush=True)
    rc = proc.wait()
    dt = time.time() - t0
    print("[rc=%d] %.1f초" % (rc, dt), flush=True)
    if rc != 0:
        print("실패했습니다. 같은 명령에 --help 를 붙여서 인자 이름부터 확인하십시오.", flush=True)
        if check:
            raise RuntimeError("명령이 실패했습니다: " + " ".join(cmd))
    return {"rc": rc, "seconds": dt, "output": "".join(buf)}


def vqa(module, *args, **kw):
    """python -m vqa.<module> 을 실행합니다."""
    return run([PY, "-m", "vqa." + module] + [str(a) for a in args], **kw)


def script(name, *args, **kw):
    """scripts/<name> 을 실행합니다."""
    return run([PY, str(Path("scripts") / name)] + [str(a) for a in args], **kw)


def read_csv_rows(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def read_ids(path):
    return [ln.strip() for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip()]


def sha256_of(path, limit=None):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    return digest[:limit] if limit else digest


print("PROJECT_ROOT      :", PROJECT_ROOT)
for _name in ("VQA_DATA_ROOT", "VQA_RUN_ROOT", "VQA_LOCAL_SCRATCH", "VQA_MODEL_ROOT"):
    print("%-18s: %s" % (_name, os.environ.get(_name, "(미설정)")))

In [ ]:
# --- 이미지 미리보기 헬퍼 (표시 전용) ---
import base64, io
from IPython.display import HTML, display
from PIL import Image

Image.MAX_IMAGE_PIXELS = None


def thumb_b64(path, size=380):
    im = Image.open(path).convert("RGB")
    im.thumbnail((size, size))
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=80)
    return base64.b64encode(buf.getvalue()).decode()


def show_images(paths, captions=None, size=380):
    captions = captions or [Path(p).name for p in paths]
    html = ['<div style="display:flex;gap:10px;flex-wrap:wrap;align-items:flex-start">']
    for p, cap in zip(paths, captions):
        html.append(
            '<figure style="margin:0;font-size:12px">'
            '<img src="data:image/jpeg;base64,%s" style="max-width:%dpx">'
            "<figcaption>%s</figcaption></figure>" % (thumb_b64(p, size), size, cap)
        )
    html.append("</div>")
    display(HTML("".join(html)))


print("이미지 미리보기 헬퍼를 준비했습니다.")

## 1. 데이터 검사

`scripts/audit_data.py` 가 CSV의 필수 컬럼, 경로 존재 여부, 이미지 열림 여부, 중복 ID, 근접 중복
후보를 한 번에 검사합니다. 이미지 지문을 모두 계산하므로 처음 실행할 때는 몇 분이 걸립니다.

결과로 세 개의 파일이 만들어집니다. `audit_report.json` 은 기계가 읽는 요약이고,
`audit_report.md` 는 사람이 읽는 보고서이며, `near_duplicate_pairs.csv` 는 눈으로 확인해야 할
근접 중복 후보 목록입니다.

In [ ]:
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

script("audit_data.py",
       "--data-root", os.environ["VQA_DATA_ROOT"],
       "--out-dir", OUT_DIR,
       "--threshold", PHASH_THRESHOLD)

AUDIT = json.loads(Path(AUDIT_REPORT).read_text(encoding="utf-8"))
print()
print("검사 결과 최상위 키:", sorted(AUDIT.keys()))
for split, info in (AUDIT.get("splits") or {}).items():
    print("[%s] %s" % (split, json.dumps(info, ensure_ascii=False)[:200]))
for key in ("exact_duplicates", "near_duplicates"):
    if key in AUDIT:
        print("%-18s: %s쌍" % (key, AUDIT[key].get("n_pairs")))
print()
print("사람이 읽을 보고서: artifacts/data/audit_report.md")

## 2. 근접 중복 후보를 눈으로 확인

자동으로 뽑은 후보를 사람이 확인합니다. 비슷한 간판이라는 이유만으로 전부 같은 그룹으로 묶으면
학습 데이터가 크게 줄어드니, 실제로 같은 장면을 찍은 사진인지를 기준으로 판단하십시오.

두 장이 같은 장면이면 `SAME` 목록에, 서로 다른 장면이면 `DIFFERENT` 목록에 쌍 번호를 적습니다.

In [ ]:
DUP_PAIRS = read_csv_rows(PAIRS_CSV)
print("근접 중복 후보 쌍: %d개" % len(DUP_PAIRS))
if DUP_PAIRS:
    print("컬럼:", list(DUP_PAIRS[0].keys()))

ALL_ROWS = []
for name in ("train.csv", "dev.csv", "test.csv"):
    p = Path(os.environ["VQA_DATA_ROOT"]) / name
    if p.exists():
        ALL_ROWS += read_csv_rows(p)
ID2PATH = {r["id"]: Path(os.environ["VQA_DATA_ROOT"]) / r["path"] for r in ALL_ROWS}

TRAIN_ROWS = read_csv_rows(Path(os.environ["VQA_DATA_ROOT"]) / "train.csv")
ID2ROW = {r["id"]: r for r in TRAIN_ROWS}


def pair_ids(row):
    return row["id_a"], row["id_b"]


# train 안쪽 쌍을 먼저 봅니다. split 경계를 넘는 쌍은 누수 검사에서 따로 다룹니다.
priority = [r for r in DUP_PAIRS if r.get("split_a") == "train" and r.get("split_b") == "train"]
priority += [r for r in DUP_PAIRS if r not in priority]

for idx, row in enumerate(priority[:DUP_PAIRS_TO_SHOW]):
    ia, ib = pair_ids(row)
    print("--- 쌍 %d --- %s(%s) vs %s(%s), 거리 %s"
          % (idx, ia, row.get("split_a"), ib, row.get("split_b"), row.get("distance")))
    for i in (ia, ib):
        r = ID2ROW.get(i)
        if r:
            print("     질문: %s" % r["question"][:70])
    try:
        show_images([ID2PATH[ia], ID2PATH[ib]], captions=[ia, ib], size=320)
    except Exception as exc:
        print("  이미지를 열지 못했습니다:", exc)

In [ ]:
# 위에서 확인한 결과를 쌍 번호로 적습니다. 예: SAME = [0, 3, 7]
SAME = []
DIFFERENT = []

confirm = {
    "reviewer": "수빈",
    "pairs_csv": PAIRS_CSV,
    "n_candidates": len(DUP_PAIRS),
    "same_pairs": [list(pair_ids(priority[i])) for i in SAME],
    "different_pairs": [list(pair_ids(priority[i])) for i in DIFFERENT],
    "reviewed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
CONFIRM_PATH = Path("artifacts/data/dup_confirm.json")
CONFIRM_PATH.write_text(json.dumps(confirm, ensure_ascii=False, indent=2), encoding="utf-8")
print("검수 결과를 저장했습니다:", CONFIRM_PATH)
print("같은 장면 %d쌍, 다른 장면 %d쌍" % (len(SAME), len(DIFFERENT)))
print()
print("이 파일은 사람이 남기는 검수 기록입니다. make_split.py는 이 파일을 읽지 않고,")
print("자체 threshold로 그룹을 묶습니다. 검수 결과와 자동 판정이 크게 어긋나면")
print("--threshold 값을 조정해서 다시 실행하는 방식으로 반영하십시오.")

## 3. 고정 split 생성

그룹 분리를 먼저 지키고, 그 범위 안에서 정답 분포를 맞춥니다. 같은 인자로 두 번 실행하면 같은
결과가 나오므로, 값을 바꿔 가면서 val 크기를 조정해도 됩니다.

`split_v1.json` 과 함께 `train_ids.txt`, `val_ids.txt`, `split_leakage.csv` 가 만들어집니다.
누수 파일에 행이 남아 있으면 그 내용을 먼저 확인해야 합니다.

In [ ]:
script("make_split.py",
       "--data-root", os.environ["VQA_DATA_ROOT"],
       "--out-dir", OUT_DIR,
       "--val-size", VAL_SIZE,
       "--seed", SEED,
       "--threshold", PHASH_THRESHOLD)

SPLIT = json.loads(Path(SPLIT_OUT).read_text(encoding="utf-8"))
train_ids = SPLIT["train_ids"]
val_ids = SPLIT["val_ids"]
print()
print("train %d개 / val %d개" % (len(train_ids), len(val_ids)))
overlap = set(train_ids) & set(val_ids)
print("겹치는 ID: %d개 (0이어야 합니다)" % len(overlap))
assert not overlap, "train과 val이 겹칩니다. split 생성 조건을 다시 확인하십시오."

leak_path = Path(OUT_DIR) / "split_leakage.csv"
if leak_path.exists():
    leaks = read_csv_rows(leak_path)
    print("누수 검사 결과: %d행" % len(leaks))
    for r in leaks[:10]:
        print("  ", json.dumps(r, ensure_ascii=False)[:160])

## 4. val 정답 분포 확인

한 선택지가 지나치게 많으면 정확도 해석이 왜곡됩니다. 대체로 각 선택지가 20퍼센트에서 30퍼센트
사이에 들어오면 충분합니다.

In [ ]:
from collections import Counter

val_answers = Counter(ID2ROW[i]["answer"].strip().lower() for i in val_ids if i in ID2ROW)
total = sum(val_answers.values())
print("val 정답 분포 (전체 %d문항)" % total)
for letter in ["a", "b", "c", "d"]:
    n = val_answers.get(letter, 0)
    bar = "#" * int(40 * n / max(total, 1))
    print("  %s: %4d (%5.1f%%) %s" % (letter, n, 100.0 * n / max(total, 1), bar))

train_answers = Counter(ID2ROW[i]["answer"].strip().lower() for i in train_ids if i in ID2ROW)
print()
print("train 정답 분포 (전체 %d문항)" % sum(train_answers.values()))
for letter in ["a", "b", "c", "d"]:
    n = train_answers.get(letter, 0)
    print("  %s: %5d (%5.1f%%)" % (letter, n, 100.0 * n / max(sum(train_answers.values()), 1)))

## 5. split 해시를 팀에 공유

아래 출력의 `split_sha256` 값이 모든 팀원에게서 같아야 합니다. 값이 다르면 서로 다른 데이터로
평가하고 있는 것이므로 즉시 알려야 합니다.

In [ ]:
content = json.dumps({"train_ids": sorted(train_ids), "val_ids": sorted(val_ids)},
                     ensure_ascii=False, sort_keys=True)
content_hash = hashlib.sha256(content.encode("utf-8")).hexdigest()

block = [
    "===== 고정 split 공유 =====",
    "파일        : %s" % SPLIT_OUT,
    "train 문항  : %d" % len(train_ids),
    "val 문항    : %d" % len(val_ids),
    "split_sha256: %s" % content_hash,
    "파일 sha256 : %s" % sha256_of(SPLIT_OUT),
    "근접중복검수: 같은 장면 %d쌍 / 다른 장면 %d쌍" % (len(SAME), len(DIFFERENT)),
    "다음 작업   : 창혁은 11번 노트북, 유림은 12번 노트북을 시작할 수 있습니다.",
    "==========================",
]
print("\n".join(block))